# Core ML Image Classifier Demo

This notebook demonstrates the `ImageClassifier` and `ImageIndexer` components.

In [2]:
import os
import sys

from PIL import Image

sys.path.append(os.path.abspath('.'))


from src.image_classifier import ImageClassifier
from src.image_indexer import ImageIndexer

## Configuration

In [3]:
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
CATALOGUE_PATH = os.path.join(DATA_DIR, 'catalogue.csv')
WEIGHTS_PATH = os.path.join(PROJECT_ROOT, 'models', 'mobilenet_v2.pt')

os.makedirs(IMAGES_DIR, exist_ok=True)
for name, color in {
    'red_tile.jpg': (220, 30, 30),
    'blue_tile.jpg': (30, 30, 220),
    'green_tile.jpg': (30, 220, 30),
    'yellow_tile.jpg': (220, 220, 30),
    'query.jpg': (230, 40, 40),
}.items():
    path = os.path.join(IMAGES_DIR, name)
    if not os.path.exists(path):
        Image.new('RGB', (224, 224), color).save(path)

## Image Classification Demo

In [4]:
print('=== Image Classification ===')
classifier = ImageClassifier()

for name in ['red_tile.jpg', 'blue_tile.jpg', 'green_tile.jpg']:
    path = os.path.join(IMAGES_DIR, name)
    print(f'\n{name}')
    for label, confidence in classifier.classify(path, top_k=3):
        print(f'  {confidence:.3f}  {label}')

classifier.save(WEIGHTS_PATH)
print(f'\nSaved weights to {WEIGHTS_PATH}')

=== Image Classification ===
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /Users/mac/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:04<00:00, 2.89MB/s]



red_tile.jpg
  0.013  theater curtain
  0.005  wall clock
  0.005  red wine

blue_tile.jpg
  0.013  nematode
  0.009  matchstick
  0.008  jellyfish

green_tile.jpg
  0.020  nematode
  0.009  nail
  0.008  letter opener

Saved weights to /Users/mac/MLProjects/core_ml_image_classifier/models/mobilenet_v2.pt


## Image Retrieval Demo

In [5]:
print('=== Image Retrieval ===')
items, relative_paths = ImageIndexer.from_csv(CATALOGUE_PATH, 'item', 'image')
image_paths = [os.path.join(DATA_DIR, p) for p in relative_paths]

indexer = ImageIndexer(classifier=classifier)
indexer.fit(items, image_paths)

seed = items[0]
print(f"Items visually similar to '{seed}':")
for item, score in indexer.recommend(seed, top_k=3):
    print(f'  {score:.3f}  {item}')

query_path = os.path.join(IMAGES_DIR, 'query.jpg')
print("\nRecommendations for query image 'query.jpg':")
for item, score in indexer.recommend_from_image(query_path, top_k=3):
    print(f'  {score:.3f}  {item}')

=== Image Retrieval ===
Items visually similar to 'Crimson Tile':
  0.721  Goldenrod Tile
  0.710  Emerald Tile
  0.575  Azure Tile

Recommendations for query image 'query.jpg':
  0.997  Crimson Tile
  0.737  Goldenrod Tile
  0.732  Emerald Tile
